# Spectral Newest: 30-Run Benchmark + Baseline-Aware Graphs

This notebook helps you:
- run about 30 benchmark calls,
- inspect end-of-run successness (counts + rates),
- visualize baseline and edited signals directly (no baseline-delta plotting).

## 1) Imports and plot style setup

In [1]:
import json
import re
import sys
import time
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

try:
    from scipy.fft import rfft, rfftfreq
except Exception:
    rfft = None
    rfftfreq = None

np.random.seed(42)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

FIGURES = []
print("Imports ready")

Imports ready


## 2) Benchmark matrix configuration (30 runs)

In [2]:
cwd = Path.cwd().resolve()
workspace_root = cwd if (cwd / "structural_benchmark.py").exists() else cwd.parent
analysis_dir = workspace_root / "analysis_out"
analysis_dir.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen"
N_RUNS = 1
N_TESTS_PER_CALL = 10
START_IDX = 0
RUN_BENCHMARKS = False
PYTHON_BIN = sys.executable

benchmark_plan = pd.DataFrame(
    {
        "run_id": np.arange(1, N_RUNS + 1, dtype=int),
        "start_idx": np.arange(START_IDX, START_IDX + N_RUNS, dtype=int),
        "n_tests": [N_TESTS_PER_CALL] * N_RUNS,
        "model": [MODEL_NAME] * N_RUNS,
        "trim_first_layers": [2] * N_RUNS,
        "trim_last_layers": [2] * N_RUNS,
        "spectral_top_k": [50] * N_RUNS,
    }
)

print(f"workspace_root={workspace_root}")
print(f"analysis_dir={analysis_dir}")
print(f"RUN_BENCHMARKS={RUN_BENCHMARKS}")
display(benchmark_plan.head())
print(f"Planned runs: {len(benchmark_plan)}")

workspace_root=/data/olexa/Latium
analysis_dir=/data/olexa/Latium/analysis_out
RUN_BENCHMARKS=False


,run_id,start_idx,n_tests,model,trim_first_layers,trim_last_layers,spectral_top_k
0,1,0,10,Qwen,2,2,50


Planned runs: 1


## 3) Load inputs and build baseline signal

In [3]:
SIGNAL_KEYS = [
    "sv_z_scores",
    "sv_ratio_scores",
    "sv_z_rolling_z_scores",
    "sv_ratio_rolling_z_scores",
    "pcs_composite_rank_scores",
    "sv_pcs_contradiction_scores",
    "rome_hybrid_scores",
    "pcs_next_jump_scores",
    "pcs_neighbor_var_scores",
    "pcs_next_curvature_scores",
    "pcs_cross_shift_scores",
]


def to_int_float_map(m):
    if not isinstance(m, dict):
        return {}
    out = {}
    for k, v in m.items():
        try:
            out[int(k)] = float(v)
        except Exception:
            continue
    return out


def signal_series(signal_block, key):
    block = signal_block if isinstance(signal_block, dict) else {}
    mapped = to_int_float_map(block.get(key, {}))
    if not mapped:
        return pd.Series(dtype=float)
    return pd.Series(mapped, dtype=float).sort_index()


def list_result_files(directory: Path, model_name: str | None = None):
    files = sorted(directory.glob("rome_structural_*.json"), key=lambda p: p.stat().st_mtime)
    if model_name:
        files = [p for p in files if model_name in p.name]
    return files


def load_result(path: Path):
    with open(path, "r") as f:
        return json.load(f)


existing_files = list_result_files(analysis_dir, MODEL_NAME)
latest_result_path = existing_files[-1] if existing_files else None

if latest_result_path is None:
    print("No result files found yet in analysis_out.")
    baseline_signal_store = {}
else:
    reference_result = load_result(latest_result_path)
    baseline_block = reference_result.get("baseline_spectral", {})
    baseline_signal_store = {k: signal_series(baseline_block, k) for k in SIGNAL_KEYS}
    print(f"Latest result file: {latest_result_path.name}")
    print("Baseline signals loaded:", [k for k, s in baseline_signal_store.items() if len(s) > 0])

No result files found yet in analysis_out.


## 4) Define benchmark functions and success criteria

In [4]:
def run_single_benchmark(row, output_dir: Path):
    cmd = [
        PYTHON_BIN,
        str(workspace_root / "structural_benchmark.py"),
        "--model",
        str(row["model"]),
        "--n-tests",
        str(int(row["n_tests"])),
        "--start-idx",
        str(int(row["start_idx"])),
        "--output-dir",
        str(output_dir),
        "--spectral-top-k",
        str(int(row["spectral_top_k"])),
        "--trim-first-layers",
        str(int(row["trim_first_layers"])),
        "--trim-last-layers",
        str(int(row["trim_last_layers"])),
    ]

    t0 = time.perf_counter()
    proc = subprocess.run(
        cmd,
        cwd=workspace_root,
        text=True,
        capture_output=True,
    )
    runtime_sec = time.perf_counter() - t0

    merged_log = (proc.stdout or "") + "\n" + (proc.stderr or "")
    match = re.findall(r"Saved to:\s*(.+\.json)", merged_log)
    result_path = Path(match[-1].strip()) if match else None

    return {
        "run_id": int(row["run_id"]),
        "start_idx": int(row["start_idx"]),
        "returncode": int(proc.returncode),
        "runtime_sec": float(runtime_sec),
        "result_path": str(result_path) if result_path else None,
        "ok": bool(proc.returncode == 0 and result_path and result_path.exists()),
        "stdout_tail": "\n".join((proc.stdout or "").splitlines()[-20:]),
        "stderr_tail": "\n".join((proc.stderr or "").splitlines()[-20:]),
    }


def extract_test_rows(result_obj, result_path: str, run_id: int | None = None):
    rows = []
    target_layer = int(result_obj.get("metadata", {}).get("target_layer", -1))
    tests = result_obj.get("tests", [])

    for idx, test in enumerate(tests):
        if test.get("skipped"):
            continue

        spectral = test.get("spectral_detection") or {}
        normal = test.get("normal_detection") or {}
        blind = test.get("blind_detection") or {}
        ipr_det = (test.get("ipr") or {}).get("ipr_detection") or {}
        acc = test.get("accuracy") or {}

        row = {
            "run_id": run_id,
            "result_path": result_path,
            "test_idx": idx,
            "subject": test.get("subject"),
            "target_layer": target_layer,
            "rome_success": bool(acc.get("rome_success", False)),
            "normal_layer": normal.get("anomalous_layer"),
            "blind_layer": blind.get("anomalous_layer"),
            "spectral_layer": spectral.get("anomalous_layer"),
            "spectral_hybrid_layer": spectral.get("anomalous_layer_hybrid", spectral.get("anomalous_layer")),
            "spectral_rank_layer": spectral.get("anomalous_layer_rank_fusion"),
            "ipr_layer": ipr_det.get("anomalous_layer"),
            "normal_correct": bool(acc.get("normal_correct", False)),
            "blind_correct": bool(acc.get("blind_correct", False)),
            "spectral_correct": bool(acc.get("spectral_correct", False)),
            "ipr_correct": bool(acc.get("ipr_detection_correct", False)),
        }
        row["benchmark_success"] = bool(row["rome_success"] and row["spectral_correct"])
        rows.append(row)

    return rows

print("Runner + success criteria ready")

Runner + success criteria ready


## 5) Execute all benchmark runs

In [5]:
run_logs = []

if RUN_BENCHMARKS:
    print(f"Running {len(benchmark_plan)} benchmark calls...")
    for _, row in benchmark_plan.iterrows():
        run_info = run_single_benchmark(row, analysis_dir)
        run_logs.append(run_info)
        print(
            f"run_id={run_info['run_id']:02d} start_idx={run_info['start_idx']} "
            f"ok={run_info['ok']} runtime={run_info['runtime_sec']:.1f}s"
        )
else:
    print("RUN_BENCHMARKS=False, using latest existing result files instead.")
    existing = list_result_files(analysis_dir, MODEL_NAME)
    picked = existing[-N_RUNS:] if existing else []
    for i, path in enumerate(picked, start=1):
        run_logs.append(
            {
                "run_id": i,
                "start_idx": None,
                "returncode": 0,
                "runtime_sec": np.nan,
                "result_path": str(path),
                "ok": True,
                "stdout_tail": "",
                "stderr_tail": "",
            }
        )

run_logs_df = pd.DataFrame(run_logs)
print(f"Runs loaded: {len(run_logs_df)}")
if not run_logs_df.empty:
    display(run_logs_df.head())

RUN_BENCHMARKS=False, using latest existing result files instead.
Runs loaded: 0


## 6) Assemble results table and final success summary

In [6]:
all_test_rows = []
loaded_results = {}

for _, run in run_logs_df.iterrows():
    result_path = run.get("result_path")
    if not result_path:
        continue
    result_path = Path(result_path)
    if not result_path.exists():
        continue

    obj = load_result(result_path)
    loaded_results[str(result_path)] = obj
    rows = extract_test_rows(obj, str(result_path), run_id=int(run.get("run_id", -1)))
    all_test_rows.extend(rows)

results_df = pd.DataFrame(all_test_rows)

if results_df.empty:
    print("No benchmark test rows found. Run section 5 first or ensure analysis_out has result JSONs.")
else:
    detector_cols = ["normal_correct", "blind_correct", "spectral_correct", "ipr_correct"]

    total_tests = int(len(results_df))
    total_pass = int(results_df["benchmark_success"].sum())
    success_rate = total_pass / total_tests if total_tests else 0.0

    summary_table = pd.DataFrame(
        {
            "metric": [
                "Total tests",
                "Benchmark success (ROME & Spectral)",
                "Benchmark success rate",
                "ROME success rate",
                "Normal detection accuracy",
                "Blind detection accuracy",
                "Spectral detection accuracy",
                "IPR detection accuracy",
            ],
            "value": [
                total_tests,
                total_pass,
                f"{success_rate:.1%}",
                f"{results_df['rome_success'].mean():.1%}",
                f"{results_df['normal_correct'].mean():.1%}",
                f"{results_df['blind_correct'].mean():.1%}",
                f"{results_df['spectral_correct'].mean():.1%}",
                f"{results_df['ipr_correct'].mean():.1%}",
            ],
        }
    )

    print("Final successness summary")
    display(summary_table)

    per_run = (
        results_df.groupby("run_id", dropna=False)
        .agg(
            tests=("subject", "count"),
            benchmark_success_rate=("benchmark_success", "mean"),
            rome_success_rate=("rome_success", "mean"),
            spectral_accuracy=("spectral_correct", "mean"),
        )
        .reset_index()
    )
    display(per_run.head(10))

    cols = [
        "run_id",
        "subject",
        "target_layer",
        "normal_layer",
        "blind_layer",
        "spectral_layer",
        "spectral_hybrid_layer",
        "spectral_rank_layer",
        "ipr_layer",
        "rome_success",
        "normal_correct",
        "blind_correct",
        "spectral_correct",
        "ipr_correct",
        "benchmark_success",
    ]
    display(results_df[cols].head(15))

No benchmark test rows found. Run section 5 first or ensure analysis_out has result JSONs.


## 7) Plot frequency-domain graphs with baseline as separate signal

In [7]:
def gather_signal_arrays(signal_key, loaded_results_dict):
    baseline_series = None
    edited_series_list = []

    for obj in loaded_results_dict.values():
        base_block = obj.get("baseline_spectral", {})
        bs = signal_series(base_block, signal_key)
        if len(bs) > 0 and baseline_series is None:
            baseline_series = bs

        for test in obj.get("tests", []):
            if test.get("skipped"):
                continue
            ss = signal_series(test.get("spectral_detection", {}), signal_key)
            if len(ss) > 0:
                edited_series_list.append(ss)

    if baseline_series is None and not edited_series_list:
        return None, None, None

    idx_sets = []
    if baseline_series is not None:
        idx_sets.append(set(baseline_series.index.tolist()))
    idx_sets.extend(set(s.index.tolist()) for s in edited_series_list)
    layers = sorted(set().union(*idx_sets))

    baseline = None
    if baseline_series is not None:
        baseline = baseline_series.reindex(layers).astype(float).to_numpy()

    edited = None
    if edited_series_list:
        edited = np.vstack([s.reindex(layers).astype(float).to_numpy() for s in edited_series_list])

    return np.array(layers, dtype=int), baseline, edited


if not loaded_results:
    print("No loaded results available for plotting.")
elif rfft is None or rfftfreq is None:
    print("scipy.fft not available; install scipy for frequency-domain plots.")
else:
    fig, axes = plt.subplots(5, 2, figsize=(16, 18))
    axes = axes.flatten()

    for ax, key in zip(axes, SIGNAL_KEYS):
        layers, baseline, edited = gather_signal_arrays(key, loaded_results)
        if layers is None or baseline is None or edited is None or edited.size == 0:
            ax.set_title(f"{key} (missing)")
            ax.axis("off")
            continue

        edited_mean = np.nanmean(edited, axis=0)

        yb = np.nan_to_num(baseline, nan=0.0)
        ye = np.nan_to_num(edited_mean, nan=0.0)

        n = len(yb)
        xf = rfftfreq(n, d=1.0)
        fb = np.abs(rfft(yb))
        fe = np.abs(rfft(ye))

        ax.plot(xf, fb, color="black", linewidth=2, label="Baseline")
        ax.plot(xf, fe, color="tab:blue", linewidth=2, label="Edited mean")
        ax.set_title(f"FFT magnitude: {key}")
        ax.set_xlabel("Frequency")
        ax.set_ylabel("Magnitude")
        ax.legend(loc="upper right", fontsize=8)

    for ax in axes[len(SIGNAL_KEYS):]:
        ax.axis("off")

    plt.tight_layout()
    FIGURES.append(("freq_domain_signals", fig))
    plt.show()

No loaded results available for plotting.


## 8) Plot time-domain graphs (raw baseline + raw edited)

In [8]:
if results_df.empty or not loaded_results:
    print("No data available for time-domain plots.")
else:
    target_layer = int(results_df["target_layer"].mode().iloc[0]) if "target_layer" in results_df else None

    for key in SIGNAL_KEYS:
        layers, baseline, edited = gather_signal_arrays(key, loaded_results)
        if layers is None or baseline is None or edited is None or edited.size == 0:
            continue

        edited_mean = np.nanmean(edited, axis=0)

        fig, ax = plt.subplots(1, 1, figsize=(12, 4.5))

        # Raw signals only: baseline + edited traces + edited mean
        for row in edited:
            ax.plot(layers, row, color="lightgray", alpha=0.35, linewidth=1)
        ax.plot(layers, baseline, color="black", linewidth=2, label="Baseline")
        ax.plot(layers, edited_mean, color="tab:blue", linewidth=2, label="Edited mean")
        if target_layer is not None:
            ax.axvline(target_layer, color="tab:red", linestyle="--", linewidth=1.5, label=f"Target L{target_layer}")
        ax.set_title(f"Signal profile: {key}")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Signal value")
        ax.legend(loc="best", fontsize=8)

        plt.tight_layout()
        FIGURES.append((f"time_domain_{key}", fig))
        plt.show()

No data available for time-domain plots.


## 9) Plot benchmark success, stability, and runtime metrics

In [9]:
if results_df.empty:
    print("No results to plot success/stability/runtime metrics.")
else:
    detector_rate = pd.Series(
        {
            "ROME success": results_df["rome_success"].mean(),
            "Normal": results_df["normal_correct"].mean(),
            "Blind": results_df["blind_correct"].mean(),
            "Spectral": results_df["spectral_correct"].mean(),
            "IPR": results_df["ipr_correct"].mean(),
            "Benchmark success": results_df["benchmark_success"].mean(),
        }
    ).sort_values(ascending=False)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # (a) success-rate bar
    ax = axes[0, 0]
    detector_rate.plot(kind="bar", ax=ax, color="tab:blue")
    ax.set_ylim(0, 1)
    ax.set_title("Success rate by method")
    ax.set_ylabel("Rate")

    # (b) pass/fail counts
    ax = axes[0, 1]
    pass_fail = pd.Series({"Pass": int(results_df["benchmark_success"].sum()), "Fail": int((~results_df["benchmark_success"]).sum())})
    pass_fail.plot(kind="bar", ax=ax, color=["tab:green", "tab:red"])
    ax.set_title("Benchmark pass/fail counts")
    ax.set_ylabel("Count")

    # (c) runtime distribution
    ax = axes[1, 0]
    if not run_logs_df.empty and run_logs_df["runtime_sec"].notna().any():
        sns.boxplot(data=run_logs_df, y="runtime_sec", ax=ax, color="tab:purple")
        ax.set_title("Runtime distribution per benchmark call")
        ax.set_ylabel("Seconds")
    else:
        ax.text(0.5, 0.5, "No runtime data (RUN_BENCHMARKS=False)", ha="center", va="center")
        ax.set_axis_off()

    # (d) correctness heatmap
    ax = axes[1, 1]
    heat_cols = ["normal_correct", "blind_correct", "spectral_correct", "ipr_correct", "benchmark_success"]
    heat_df = results_df[heat_cols].astype(int)
    sns.heatmap(heat_df.T, cmap="YlGnBu", cbar=True, ax=ax)
    ax.set_title("Per-test correctness heatmap")
    ax.set_xlabel("Test row index")

    plt.tight_layout()
    FIGURES.append(("summary_success_runtime", fig))
    plt.show()

    # Predicted layer histograms per detector
    pred_cols = ["normal_layer", "blind_layer", "spectral_layer", "spectral_hybrid_layer", "spectral_rank_layer", "ipr_layer"]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()
    target_layer = int(results_df["target_layer"].mode().iloc[0])

    for ax, col in zip(axes, pred_cols):
        vals = pd.to_numeric(results_df[col], errors="coerce").dropna()
        if vals.empty:
            ax.set_title(f"{col} (no data)")
            ax.set_axis_off()
            continue
        ax.hist(vals, bins=min(20, max(5, vals.nunique())), color="tab:gray", edgecolor="black", alpha=0.85)
        ax.axvline(target_layer, color="tab:red", linestyle="--", linewidth=2, label=f"Target L{target_layer}")
        ax.set_title(f"Predicted layers: {col}")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Count")
        ax.legend(fontsize=8)

    for ax in axes[len(pred_cols):]:
        ax.axis("off")

    plt.tight_layout()
    FIGURES.append(("predicted_layer_histograms", fig))
    plt.show()

No results to plot success/stability/runtime metrics.


In [10]:
# Hybrid blind score profile per layer (no baseline subtraction)
if not loaded_results:
    print("No loaded results available for hybrid score plot.")
else:
    layers, baseline, edited = gather_signal_arrays("rome_hybrid_scores", loaded_results)
    if layers is None or edited is None or edited.size == 0:
        print("No rome_hybrid_scores found in loaded results.")
    else:
        edited_mean = np.nanmean(edited, axis=0)
        q25 = np.nanpercentile(edited, 25, axis=0)
        q75 = np.nanpercentile(edited, 75, axis=0)

        fig, ax = plt.subplots(1, 1, figsize=(12, 4.8))

        for row in edited:
            ax.plot(layers, row, color="lightgray", alpha=0.25, linewidth=1)

        if baseline is not None:
            ax.plot(layers, baseline, color="black", linewidth=2, label="Baseline")

        ax.plot(layers, edited_mean, color="tab:blue", linewidth=2.2, label="Edited mean")
        ax.fill_between(layers, q25, q75, color="tab:blue", alpha=0.18, label="Edited IQR (25-75%)")

        if 'results_df' in globals() and not results_df.empty and 'target_layer' in results_df:
            target_layer = int(results_df["target_layer"].mode().iloc[0])
            ax.axvline(target_layer, color="tab:red", linestyle="--", linewidth=1.6, label=f"Target L{target_layer}")

        ax.set_title("Hybrid blind score per layer (rome_hybrid_scores)")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Hybrid blind score")
        ax.legend(loc="best", fontsize=8)
        plt.tight_layout()

        FIGURES.append(("hybrid_blind_score_per_layer", fig))
        plt.show()

No loaded results available for hybrid score plot.


In [11]:
    # Spread plot (distribution) for hybrid score per layer: baseline vs edited, no deltas
if not loaded_results:
    print("No loaded results available for spread plot.")
else:
    layers, baseline, edited = gather_signal_arrays("rome_hybrid_scores", loaded_results)
    if layers is None or edited is None or edited.size == 0:
        print("No rome_hybrid_scores found in loaded results.")
    else:
        arr = np.asarray(edited, dtype=float)
        positions = np.arange(len(layers))

        # One distribution per layer (edited runs)
        per_layer = [arr[:, i][np.isfinite(arr[:, i])] for i in range(arr.shape[1])]
        fig, ax = plt.subplots(1, 1, figsize=(14, 5.2))

        vp = ax.violinplot(
            per_layer,
            positions=positions,
            widths=0.82,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body in vp["bodies"]:
            body.set_facecolor("tab:blue")
            body.set_edgecolor("none")
            body.set_alpha(0.18)

        # Edited quantile band + median (not delta)
        q25 = np.nanpercentile(arr, 25, axis=0)
        q75 = np.nanpercentile(arr, 75, axis=0)
        med = np.nanmedian(arr, axis=0)
        ax.fill_between(positions, q25, q75, color="tab:blue", alpha=0.20, label="Edited IQR (25-75%)")
        ax.plot(positions, med, color="tab:blue", linewidth=2.0, label="Edited median")

        # Baseline overlaid directly
        if baseline is not None:
            ax.plot(positions, baseline, color="black", linewidth=2.0, label="Baseline")
            ax.scatter(positions, baseline, color="black", s=12)

        if 'results_df' in globals() and not results_df.empty and 'target_layer' in results_df:
            target_layer = int(results_df["target_layer"].mode().iloc[0])
            if target_layer in layers:
                target_pos = int(np.where(layers == target_layer)[0][0])
                ax.axvline(target_pos, color="tab:red", linestyle="--", linewidth=1.5, label=f"Target L{target_layer}")

        ax.set_title("Spread of hybrid score per layer (baseline + edited, no deltas)")
        ax.set_xlabel("Layer")
        ax.set_ylabel("rome_hybrid_scores")

        step = 2 if len(layers) > 24 else 1
        ax.set_xticks(positions[::step])
        ax.set_xticklabels(layers[::step])
        ax.legend(loc="best", fontsize=8)
        plt.tight_layout()

        FIGURES.append(("hybrid_blind_score_spread_per_layer", fig))
        plt.show()

No loaded results available for spread plot.


## 10) Save artifacts (CSV + figures)

In [12]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
artifact_dir = analysis_dir / "spectral_newest_artifacts" / timestamp
artifact_dir.mkdir(parents=True, exist_ok=True)

if not results_df.empty:
    results_csv = artifact_dir / "benchmark_rows.csv"
    results_df.to_csv(results_csv, index=False)
    print(f"Saved rows: {results_csv}")

if not run_logs_df.empty:
    run_logs_csv = artifact_dir / "run_logs.csv"
    run_logs_df.to_csv(run_logs_csv, index=False)
    print(f"Saved run logs: {run_logs_csv}")

saved_figs = []
for name, fig in FIGURES:
    out = artifact_dir / f"{name}.png"
    fig.savefig(out, dpi=140, bbox_inches="tight")
    saved_figs.append(out)

print(f"Saved figures: {len(saved_figs)}")
for path in saved_figs[:10]:
    print(" -", path.name)

if not results_df.empty:
    total_tests = len(results_df)
    total_pass = int(results_df["benchmark_success"].sum())
    print("\n=== FINAL SUCCESSNESS ===")
    print(f"Total tests: {total_tests}")
    print(f"Benchmark success (ROME & Spectral): {total_pass}/{total_tests} ({total_pass/total_tests:.1%})")
    print(f"ROME success: {results_df['rome_success'].mean():.1%}")
    print(f"Spectral success: {results_df['spectral_correct'].mean():.1%}")
    print(f"Normal success: {results_df['normal_correct'].mean():.1%}")
    print(f"Blind success: {results_df['blind_correct'].mean():.1%}")
    print(f"IPR success: {results_df['ipr_correct'].mean():.1%}")
else:
    print("No results available yet.")

print(f"Artifacts directory: {artifact_dir}")

Saved figures: 0
No results available yet.
Artifacts directory: /data/olexa/Latium/analysis_out/spectral_newest_artifacts/20260324_110810
